In [19]:
import pandas as pd
import requests
from io import StringIO

In [20]:
df_nger = pd.read_csv("NGER.ID0243.csv")

df_accredited = pd.read_csv("power-stations-and-projects-accredited.csv")
df_committed = pd.read_csv("power-stations-and-projects-committed.csv")
df_probable = pd.read_csv("power-stations-and-projects-probable.csv")

abs_url = "https://data.api.abs.gov.au/rest/data/ABS,ABS_REGIONAL_ASGS2021,/..AUS.A?startPeriod=2020&dimensionAtObservation=AllDimensions"

headers = {
    "Accept": "application/vnd.sdmx.data+csv;labels=both"
}

response = requests.get(abs_url, headers=headers, timeout=60)
response.raise_for_status()

df_abs = pd.read_csv(StringIO(response.text))

## merge power-stations-and-projects data

In [21]:
# 1. 去掉列名空格
df_accredited.columns = df_accredited.columns.str.strip()
df_committed.columns = df_committed.columns.str.strip()

# 2. rename
df_accredited_full = df_accredited.rename(columns={
    "Power station name": "project_name",
    "State": "state",
    "Installed capacity (MW)": "capacity_mw",
    "Fuel Source (s)": "fuel_source",
    "Accreditation code": "accreditation_code",
    "Postcode": "postcode",
    "Accreditation start date": "accreditation_start_date",
    "Approval date": "approval_date"
})

df_committed_full = df_committed.rename(columns={
    "Project Name": "project_name",
    "State": "state",
    "MW Capacity": "capacity_mw",
    "Fuel Source": "fuel_source",
    "Committed Date (Month/Year)": "committed_date"
})

# 3. 直接 concat（自动对齐列，不存在的补 NaN）
df_cer = pd.concat([df_accredited_full, df_committed_full], ignore_index=True)

# 4. 加一个状态列（可选，但建议）
df_cer.loc[df_cer["accreditation_code"].notna(), "project_status"] = "accredited"
df_cer.loc[df_cer["committed_date"].notna(), "project_status"] = "committed"



In [23]:
# =========================
# NGER
# =========================
print("=== NGER shape ===")
print(df_nger.shape)

print("=== NGER columns ===")
display(df_nger.columns)

print("=== NGER head ===")
display(df_nger.head())


# =========================
# CER（你已经合并好的 df_cer）
# =========================
print("=== CER shape ===")
print(df_cer.shape)

print("=== CER columns ===")
display(df_cer.columns)

print("=== CER head ===")
display(df_cer.head())


# =========================
# ABS
# =========================
print("=== ABS shape ===")
print(df_abs.shape)

print("=== ABS columns ===")
display(df_abs.columns)

print("=== ABS head ===")
display(df_abs.head())


=== NGER shape ===
(775, 14)
=== NGER columns ===


Index(['Reporting entity', 'Facility name', 'Type', 'State',
       'Electricity production GJ', 'Electricity production MWh',
       'Total scope 1 emissions t CO2 e', 'Total scope 2 emissions t CO2 e',
       'Total emissions t CO2 e', 'Emission intensity t CO2 e MWh',
       'Grid connected', 'Grid', 'Primary fuel', 'Important notes'],
      dtype='object')

=== NGER head ===


,Reporting entity,Facility name,Type,State,Electricity production GJ,Electricity production MWh,Total scope 1 emissions t CO2 e,Total scope 2 emissions t CO2 e,Total emissions t CO2 e,Emission intensity t CO2 e MWh,Grid connected,Grid,Primary fuel,Important notes
0,ACCIONA ENERGY OCEANIA PTY LTD,Cathedral Rocks Wind Farm,F,SA,481948,133874,57,127.0,184,0.0,On,NEM,Wind,-
1,ACCIONA ENERGY OCEANIA PTY LTD,Gunning Wind Farm,F,NSW,491409,136502,50,218.0,268,0.0,On,NEM,Wind,-
2,ACCIONA ENERGY OCEANIA PTY LTD,Mortlake South Wind Farm,F,VIC,1019352,283153,202,1128.0,1330,0.0,On,NEM,Wind,-
3,ACCIONA ENERGY OCEANIA PTY LTD,Mt Gellibrand Wind Farm,F,VIC,1025451,284847,99,1273.0,1372,0.0,On,NEM,Wind,-
4,ACCIONA ENERGY OCEANIA PTY LTD,Waubra Wind Farm,F,VIC,1954964,543046,186,1114.0,1300,0.0,On,NEM,Wind,-


=== CER shape ===
(72, 10)
=== CER columns ===


...

,accreditation_code,project_name,state,postcode,capacity_mw,fuel_source,accreditation_start_date,approval_date,committed_date,project_status
0,SRPXQLO8,DONS FORT - SOLAR W SGU - QLD,QLD,4660.0,0.421,Solar,21/10/2025,6/01/2026,NaN,accredited
1,SRPYNSK0,Mintus Coles - Solar w SGU- NSW,NSW,2153.0,0.190,Solar,9/12/2025,6/01/2026,NaN,accredited
2,SRPXQLP1,Ventora - Acacia Ridge - Solar - QLD,QLD,4110.0,0.162,Solar,17/11/2025,6/01/2026,NaN,accredited
3,SRPYNSK3,Bayside Aged Care - Solar - Morisset - NSW,NSW,2264.0,0.140,Solar,16/12/2025,8/01/2026,NaN,accredited
4,SRPVWAQ0,Matrix Composites and Engineering Pty Ltd - So...,WA,6166.0,0.850,Solar,17/12/2025,8/01/2026,NaN,accredited


=== ABS shape ===
(1693, 11)
=== ABS columns ===


Index(['DATAFLOW', 'MEASURE: Data Item', 'REGIONTYPE: Region Type',
       'ASGS_2021: Region', 'FREQUENCY: Frequency', 'TIME_PERIOD: Time Period',
       'OBS_VALUE', 'UNIT_MEASURE: Unit of Measure',
       'UNIT_MULT: Unit of Multiplier', 'OBS_STATUS: Observation Status',
       'OBS_COMMENT: Observation Comment'],
      dtype='object')

=== ABS head ===


,DATAFLOW,MEASURE: Data Item,REGIONTYPE: Region Type,ASGS_2021: Region,FREQUENCY: Frequency,TIME_PERIOD: Time Period,OBS_VALUE,UNIT_MEASURE: Unit of Measure,UNIT_MULT: Unit of Multiplier,OBS_STATUS: Observation Status,OBS_COMMENT: Observation Comment
0,ABS:ABS_REGIONAL_ASGS2021(1.5.0),VEHIC_2: Households with no motor vehicles (no.),AUS: Australia,AUS: Australia,A: Annual,2021,673969.0,NUM: Number,NaN,NaN,NaN
1,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ING_TTYPE1: Aboriginal and Torres Strait Islan...,AUS: Australia,AUS: Australia,A: Annual,2021,280385.0,PSNS: Persons,NaN,NaN,NaN
2,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ERP_P_38: Estimated resident population:Person...,AUS: Australia,AUS: Australia,A: Annual,2020,2.0,PCT: Percent,NaN,NaN,NaN
3,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ERP_P_38: Estimated resident population:Person...,AUS: Australia,AUS: Australia,A: Annual,2021,2.1,PCT: Percent,NaN,NaN,NaN
4,ABS:ABS_REGIONAL_ASGS2021(1.5.0),ERP_P_38: Estimated resident population:Person...,AUS: Australia,AUS: Australia,A: Annual,2022,2.1,PCT: Percent,NaN,NaN,NaN


In [25]:
import pandas as pd
import re

def clean_name(x):
    if pd.isna(x):
        return ""
    x = str(x).lower().strip()

    # remove state abbreviations only
    x = re.sub(r"\b(nsw|qld|vic|wa|sa|tas|act|nt)\b", " ", x)

    # remove company suffixes only
    x = re.sub(r"\b(pty ltd|pty|ltd)\b", " ", x)

    # replace punctuation with space
    x = re.sub(r"[^a-z0-9]", " ", x)

    # compress spaces
    x = re.sub(r"\s+", " ", x).strip()

    return x

def token_set(x):
    if not x:
        return set()
    return set(x.split())

df_nger["name_key"] = df_nger["Facility name"].apply(clean_name)
df_cer["name_key"] = df_cer["project_name"].apply(clean_name)

matches = []

for _, n_row in df_nger.iterrows():
    n_name = n_row["name_key"]
    n_state = n_row["State"]
    n_tokens = token_set(n_name)

    if not n_tokens:
        continue

    for _, c_row in df_cer.iterrows():
        c_name = c_row["name_key"]
        c_state = c_row["state"]
        c_tokens = token_set(c_name)

        if n_state != c_state:
            continue
        if not c_tokens:
            continue

        overlap = n_tokens & c_tokens

        # require at least 2 shared tokens
        if len(overlap) >= 2:
            matches.append({
                "nger_name": n_row["Facility name"],
                "nger_key": n_name,
                "cer_name": c_row["project_name"],
                "cer_key": c_name,
                "state": n_state,
                "shared_tokens": ", ".join(sorted(overlap)),
                "overlap_count": len(overlap)
            })

df_match = pd.DataFrame(matches)

print("=== 匹配结果示例 ===")
display(df_match.sort_values("overlap_count", ascending=False).head(20))

print("\n=== 匹配统计 ===")
print("NGER 总数:", len(df_nger))
print("CER 总数:", len(df_cer))
print("匹配对数:", len(df_match))

if len(df_match) > 0:
    print("匹配到的 NGER 数:", df_match["nger_name"].nunique())
    print("匹配到的 CER 数:", df_match["cer_name"].nunique())

=== 匹配结果示例 ===


,nger_name,nger_key,cer_name,cer_key,state,shared_tokens,overlap_count
6,New England Solar Farm,new england solar farm,New England Solar Farm - Stage 2,new england solar farm stage 2,NSW,"england, farm, new, solar",4
167,Oakey 2 Solar Farm,oakey 2 solar farm,Wandoan South Solar Farm (Stage 2),wandoan south solar farm stage 2,QLD,"2, farm, solar",3
100,Warradarge Wind Farm,warradarge wind farm,Warradarge Wind Farm Stage 2,warradarge wind farm stage 2,WA,"farm, warradarge, wind",3
200,"Lake Bonney Stage 1,2 & 3 Wind Farm",lake bonney stage 1 2 3 wind farm,Bungama Solar Farm (stage 3),bungama solar farm stage 3,SA,"3, farm, stage",3
419,Wandoan Solar Farm 1,wandoan solar farm 1,Wandoan South Solar Farm (Stage 2),wandoan south solar farm stage 2,QLD,"farm, solar, wandoan",3
369,Clarke Creek Wind Farm,clarke creek wind farm,Lotus Creek Wind Farm,lotus creek wind farm,QLD,"creek, farm, wind",3
370,Clarke Creek Wind Farm,clarke creek wind farm,Boulder Creek Wind Farm,boulder creek wind farm,QLD,"creek, farm, wind",3
316,Starfish Hill Wind Farm,starfish hill wind farm,Carmody's Hill Wind Farm,carmody s hill wind farm,SA,"farm, hill, wind",3
284,KABAN WIND FARM PTY LTD,kaban wind farm,Boulder Creek Wind Farm,boulder creek wind farm,QLD,"farm, wind",2
285,KABAN WIND FARM PTY LTD,kaban wind farm,Wambo Wind Farm Stage 2,wambo wind farm stage 2,QLD,"farm, wind",2



=== 匹配统计 ===
NGER 总数: 775
CER 总数: 72
匹配对数: 442
匹配到的 NGER 数: 167
匹配到的 CER 数: 34
